# TradingAgents with Databricks Integration Test

This notebook tests the TradingAgents framework with Databricks Claude Sonnet 4.5.

All output will be saved to markdown files in the `analysis_results/{symbol}/` directory with the format: `{symbol}_{timestamp}.md`

In [10]:
import os
import sys
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from tradingagents.graph.trading_graph import TradingAgentsGraph
from tradingagents.default_config import DEFAULT_CONFIG

In [11]:
# Create a custom class to capture stdout to both console and file
class OutputCapture:
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log_file = open(filename, 'w')

    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def close(self):
        self.log_file.close()
        sys.stdout = self.terminal

In [12]:
# Load environment variables from .env file
load_dotenv()

# Check if Databricks credentials are set
if not os.getenv("DATABRICKS_TOKEN"):
    print("ERROR: DATABRICKS_TOKEN environment variable is not set")
    print("Please set it in your .env file")
else:
    print("✓ DATABRICKS_TOKEN is set")

if not os.getenv("DATABRICKS_BASE_URL"):
    print("ERROR: DATABRICKS_BASE_URL environment variable is not set")
    print("Please set it in your .env file")
else:
    print("✓ DATABRICKS_BASE_URL is set")

✓ DATABRICKS_TOKEN is set
✓ DATABRICKS_BASE_URL is set


In [13]:
# Create a custom config
config = DEFAULT_CONFIG.copy()
config["llm_provider"] = "databricks"
config["deep_think_llm"] = "databricks-claude-sonnet-4-5"  # Specify your Databricks model name here
config["quick_think_llm"] = "llama_v3_3_70b_instruct_pro" # And here

# Fix the base URL to include /serving-endpoints
if config["databricks_base_url"] and not config["databricks_base_url"].endswith("/serving-endpoints"):
    config["databricks_base_url"] = config["databricks_base_url"].rstrip("/") + "/serving-endpoints"

print("Configuration:")
print(f"  LLM Provider: {config['llm_provider']}")
print(f"  Deep Think Model: {config['deep_think_llm']}")
print(f"  Quick Think Model: {config['quick_think_llm']}")
print(f"  Databricks Base URL: {config['databricks_base_url']}")
print(f"  Databricks Token: {'*' * 20 if config['databricks_token'] else 'NOT SET'}")

Configuration:
  LLM Provider: databricks
  Deep Think Model: databricks-claude-sonnet-4-5
  Quick Think Model: llama_v3_3_70b_instruct_pro
  Databricks Base URL: https://adb-8333330282859393.13.azuredatabricks.net/serving-endpoints
  Databricks Token: ********************


In [14]:
# Initialize with custom config
print("Initializing TradingAgentsGraph...")
ta = TradingAgentsGraph(debug=True, config=config)
print("✓ Initialization successful!")

Initializing TradingAgentsGraph...
✓ Initialization successful!


In [15]:
# # Set up parameters - CHANGE THESE AS NEEDED

# Create output directory structure: analysis_results/{symbol}/
def setup_output(symbol, timestamp):
    output_dir = Path("analysis_results") / symbol
    output_dir.mkdir(parents=True, exist_ok=True)

    # Create markdown filename: {symbol}_{timestamp}.md
    output_file = output_dir / f"{symbol}_{timestamp}.md"

    print(f"Output will be saved to: {output_file}")
    print("="*80)

In [16]:
# # Run analysis with output capture
# capture = OutputCapture(output_file)
# sys.stdout = capture

# try:
#     # Write markdown header
#     print(f"# Trading Analysis Report")
#     print(f"\n**Symbol:** {symbol}")
#     print(f"**Trade Date:** {trade_date}")
#     print(f"**Analysis Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
#     print(f"**LLM Provider:** {config['llm_provider']}")
#     print(f"**Model:** {config['quick_think_llm']}")
#     print("\n" + "="*80 + "\n")

#     # Run propagation
#     print(f"## Running Analysis for {symbol} on {trade_date}")
#     print("\n")

#     _, decision = ta.propagate(symbol, trade_date)

#     print("\n" + "="*80)
#     print("\n## Final Trading Decision\n")
#     print(decision)
#     print("\n" + "="*80)

# finally:
#     # Restore stdout
#     capture.close()

# print(f"\n✓ Analysis complete! Results saved to: {output_file}")

In [17]:
# # Display the final decision
# print(f"\nFinal Decision for {symbol}:")
# print("-" * 80)
# print(decision)
# print("-" * 80)

## Run Analysis for Multiple Symbols

You can run the analysis for multiple stocks in a loop:

In [18]:
# Optional: Analyze multiple symbols
symbols = ["AAPL", "MSFT", "GOOGL"]
trade_date = datetime.today().strftime('%Y-%m-%d')

for symbol in symbols:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    setup_output(symbol, timestamp)
    
    # Create output directory structure: analysis_results/{symbol}/
    output_dir = Path("analysis_results") / symbol
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create markdown filename: {symbol}_{timestamp}.md
    output_file = output_dir / f"{symbol}_{timestamp}.md"
    
    print(f"\nAnalyzing {symbol}...")
    
    capture = OutputCapture(output_file)
    sys.stdout = capture
    
    try:
        print(f"# Trading Analysis Report")
        print(f"\n**Symbol:** {symbol}")
        print(f"**Trade Date:** {trade_date}")
        print(f"**Analysis Date:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"**LLM Provider:** {config['llm_provider']}")
        print(f"**Model:** {config['quick_think_llm']}")
        print("\n" + "="*80 + "\n")
        
        print(f"## Running Analysis for {symbol} on {trade_date}")
        print("\n")
        
        _, decision = ta.propagate(symbol, trade_date)
        
        print("\n" + "="*80)
        print("\n## Final Trading Decision\n")
        print(decision)
        print("\n" + "="*80)
        
    finally:
        capture.close()
    
    print(f"✓ {symbol} analysis saved to: {output_file}")

Output will be saved to: analysis_results/AAPL/AAPL_20251028_123217.md

Analyzing AAPL...
# Trading Analysis Report

**Symbol:** AAPL
**Trade Date:** 2025-10-28
**Analysis Date:** 2025-10-28 12:32:17
**LLM Provider:** databricks
**Model:** llama_v3_3_70b_instruct_pro


## Running Analysis for AAPL on 2025-10-28


================================ Human Message =================================

AAPL
================================== Ai Message ==================================
Tool Calls:
  get_stock_data (call_1d2a012e-cb7c-4bfa-b33c-0c13475af11d)
 Call ID: call_1d2a012e-cb7c-4bfa-b33c-0c13475af11d
  Args:
    symbol: AAPL
    start_date: 2020-01-01
    end_date: 2025-10-28
DEBUG: get_stock_data - Primary: [yfinance] | Full fallback order: [yfinance → alpha_vantage → local]
DEBUG: Attempting PRIMARY vendor 'yfinance' for get_stock_data (attempt #1)
DEBUG: Calling get_YFin_data_online from vendor 'yfinance'...
SUCCESS: get_YFin_data_online from vendor 'yfinance' completed successfully